# How al-molecular works: a hands-on walkthrough

Run the cells **top to bottom** (Shift+Enter). Each section adds one idea:

1. The data: pool, embeddings, oracle
2. The score we care about: top-k recall
3. The surrogate: a cheap model that guesses scores
4. **One round of active learning, by hand**
5. The full loop: active learning vs random search
6. Uncertainty: greedy vs UCB
7. The repo's real surrogates and backbone fusion
8. Results from real runs in `al-molecular/runs/`

Sections 1–7 use small fake data, so everything runs on a CPU. They use the repo's own
variable names, so you can read `run_experiment.py` afterwards and recognize each step.

**Kernel:** `py310`, the environment al-molecular uses.

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

AL_REPO = Path("/N/slate/mengjing/repos/al-molecular")
sys.path.insert(0, str(AL_REPO))          # so we can import the repo's own code below

rng = np.random.default_rng(0)            # fixed seed -> same numbers every time you run
print("ready")

## 1. The data

In al-molecular there are three things:

| Thing | Real version | Here |
|---|---|---|
| **pool** | 2.1M SMILES strings (`molpal/libraries/EnamineHTS.csv.gz`) | 5,000 fake names |
| **embeddings** | a row of numbers per molecule, made by a big pretrained model (a *backbone*) | 8 random numbers per molecule |
| **oracle** | `{smiles: docking score}`, **lower = binds better** (`data/EnamineHTS_scores.csv.gz`) | a formula of the embedding plus noise |

In reality we pretend not to know the oracle: we may only "look up" (dock) a molecule
when the algorithm decides to spend budget on it.

In [ ]:
N_POOL = 5_000
pool_smiles = np.array([f"mol_{i}" for i in range(N_POOL)])
embeddings = rng.normal(size=(N_POOL, 8)).astype(np.float32)

e = embeddings
true_scores = (-8 + 1.2 * e[:, 0] - 0.8 * e[:, 1] + 0.6 * e[:, 2] * e[:, 3]
               + rng.normal(scale=0.4, size=N_POOL))
oracle = dict(zip(pool_smiles, true_scores))       # same shape as load_oracle() returns

print("pool_smiles :", pool_smiles[:3], "...")
print("embeddings  :", embeddings.shape, "  first row:", embeddings[0].round(2))
print("oracle      :", {str(k): round(float(oracle[k]), 2) for k in pool_smiles[:3]}, "...")

Look at the scores. Like real docking scores, most molecules are average and only a few are very good (very negative).

In [ ]:
plt.figure(figsize=(6, 3))
plt.hist(true_scores, bins=60)
plt.xlabel("docking score (lower = better)"); plt.ylabel("count"); plt.title("the whole pool")
plt.show()

## 2. The score we care about: top-k recall

"Of the truly best `k` molecules, what fraction have we docked?"
In the repo this is `true_top_k_set()` and `MVEExplorer._recall()`, with k = 1000.
We use k = 50 for the small pool.

In [ ]:
TOPK = 50
true_top_k = {s for s, _ in sorted(oracle.items(), key=lambda x: x[1])[:TOPK]}   # 50 lowest scores

def recall(labeled_scores):
    return sum(1 for s in labeled_scores if s in true_top_k) / TOPK

print("worst score inside the top-50:", round(max(oracle[s] for s in true_top_k), 2))

**Try it:** if we docked 600 molecules **at random**, about what recall should we expect?
Think about it first, then run the cell.

In [ ]:
some = rng.choice(pool_smiles, 600, replace=False)
print(f"random 600: recall = {recall({s: oracle[s] for s in some}):.0%}   (expected ~ 600/5000 = 12%)")

## 3. The surrogate

The surrogate is a cheap model: **embedding in, guessed score out**. Every surrogate in
`surrogates.py` has the same two methods, `fit(X, y)` and `predict(X)`.

Here we use plain linear regression. Notice the **sign flip**: the repo trains on
`-score` (`self._sign = -1.0` in `run_experiment.py`), so that **higher = better**.
After that, every acquisition rule simply picks the highest numbers.

In [ ]:
class LinearSurrogate:
    def fit(self, X, y):
        X1 = np.c_[X, np.ones(len(X))]                   # extra column of 1s = intercept
        self.w, *_ = np.linalg.lstsq(X1, y, rcond=None)

    def predict(self, X):
        return np.c_[X, np.ones(len(X))] @ self.w

# quick check: train on 100 random molecules, test on 1000 others
idx = rng.permutation(N_POOL)
train, test = idx[:100], idx[100:1100]
m = LinearSurrogate()
m.fit(embeddings[train], -true_scores[train])           # sign flip!
guess = m.predict(embeddings[test])

plt.figure(figsize=(4, 4))
plt.scatter(-true_scores[test], guess, s=4)
plt.xlabel("truth (sign-flipped)"); plt.ylabel("surrogate guess")
plt.title(f"correlation = {np.corrcoef(-true_scores[test], guess)[0, 1]:.2f}")
plt.show()

The guesses aren't perfect, but they're good enough to tell promising molecules from poor ones. That's all active learning needs.

## 4. One round of active learning, by hand

This is the heart of the repo, `MVEExplorer.run()`
([run_experiment.py:638-683](../../../al-molecular/run_experiment.py#L638)).
We'll go through it **one cell per step** so you can inspect every variable.

**Step 0, initial set.** Dock a random handful (`rng.choice` at run_experiment.py:617).

In [ ]:
INIT, BATCH = 100, 100
r = np.random.default_rng(42)

init_idx = r.choice(N_POOL, INIT, replace=False)
labeled_idx = set(init_idx.tolist())                                   # positions we've docked
labeled_scores = {pool_smiles[i]: oracle[pool_smiles[i]] for i in init_idx}   # {smiles: score}

print(len(labeled_idx), "docked;  recall =", f"{recall(labeled_scores):.0%}")

**Step a, train** on everything docked so far.

In [ ]:
idx = list(labeled_idx)
X_train = embeddings[idx]
y_train = -1.0 * np.array([labeled_scores[pool_smiles[i]] for i in idx])    # sign flip
model = LinearSurrogate()
model.fit(X_train, y_train)
print("trained on", X_train.shape)

**Step b, candidates** are every molecule *not* docked yet. A boolean mask does this.

In [ ]:
mask = np.ones(N_POOL, bool)
mask[idx] = False
pool_idx = np.where(mask)[0]
print(len(pool_idx), "candidates;  first few positions:", pool_idx[:8])

**Step c, predict and pick.** `np.argsort` sorts from smallest to largest, and `[::-1]` reverses
that, so the biggest come first. `top_local` holds positions *within* `pool_idx`, and
`pool_idx[top_local]` turns them back into positions in the whole pool.

In [ ]:
mu = model.predict(embeddings[pool_idx])
top_local = np.argsort(mu)[::-1][:BATCH]
selected = pool_idx[top_local]

print("best 5 guesses   :", mu[top_local[:5]].round(2))
print("their pool index :", selected[:5])
print("their TRUE score :", true_scores[selected[:5]].round(2), " (pool average is", round(true_scores.mean(), 2), ")")

**Step d, dock them** (look them up in the oracle) and add them to the labeled set.

In [ ]:
for i in selected:
    labeled_scores[pool_smiles[i]] = oracle[pool_smiles[i]]
    labeled_idx.add(int(i))

print(len(labeled_idx), "docked;  recall =", f"{recall(labeled_scores):.0%}")

After one round, recall jumped from about 2% to about 60%. A random batch of 100 would
have added roughly 2%.

**Try it:** run steps a → d again (the four cells above, not step 0). Recall keeps
climbing, but by less each time. Why do you think that is?

## 5. The full loop: active learning vs random search

The same four steps, wrapped in a function. `strategy="random"` skips the model and
picks at random, which is the baseline the repo always compares against.

In [ ]:
def run(strategy, n_rounds=5, init=100, batch=100, seed=42, make_model=LinearSurrogate):
    r = np.random.default_rng(seed)
    labeled_idx = set(r.choice(N_POOL, init, replace=False).tolist())
    curve = [len(labeled_idx & {int(s[4:]) for s in true_top_k}) / TOPK]   # recall before round 1
    model = make_model()
    for _ in range(n_rounds):
        idx = np.array(sorted(labeled_idx))
        pool_idx = np.setdiff1d(np.arange(N_POOL), idx)
        if strategy == "random":
            selected = r.choice(pool_idx, batch, replace=False)
        else:
            model.fit(embeddings[idx], -true_scores[idx])
            mu = model.predict(embeddings[pool_idx])
            selected = pool_idx[np.argsort(mu)[::-1][:batch]]
        labeled_idx.update(selected.tolist())
        curve.append(len(labeled_idx & {int(s[4:]) for s in true_top_k}) / TOPK)
    return curve

for strategy in ["random", "greedy"]:
    c = run(strategy)
    plt.plot(range(len(c)), c, marker="o", label=f"{strategy}  (final {c[-1]:.0%})")
plt.xlabel("round"); plt.ylabel(f"top-{TOPK} recall"); plt.legend(); plt.ylim(0, 1)
plt.title("same budget: 600 of 5000 docked"); plt.show()

That gap is the whole point of the repo. Everything else is about making the
active-learning curve rise faster.

**Try it:** keep the total budget fixed but change its *shape*: one big batch vs many
small ones. Smaller, more frequent batches let the model retrain more often. The effect
only shows when the budget is tight, so the cell below uses 20 initial + 100 more
molecules and averages over 10 seeds. The repo tests the same idea with batches of
0.1% / 0.2% / 0.4% and with 20 rounds instead of 5 (`adaptive_acq/`).

In [ ]:
for n_rounds, batch in [(1, 100), (2, 50), (5, 20), (10, 10)]:
    finals = [run("greedy", n_rounds=n_rounds, init=20, batch=batch, seed=s)[-1] for s in range(42, 52)]
    print(f"{n_rounds:2d} rounds x {batch:3d}:  mean final recall {np.mean(finals):.0%}")

More rounds help, but each doubling helps less than the one before. The repo found the same pattern for the number of rounds (commit f7e85b1: "increments halve per doubling").

## 6. Uncertainty: greedy vs UCB

The repo's surrogates return **two** numbers per molecule:

* `mu`: the guess
* `sigma`: how unsure the model is about that guess

An **acquisition function** turns them into one number to sort by. These are the real
functions behind `--acq greedy` and `--acq ucb`, imported straight from
`molpal/acquirer/metrics.py`:

* `greedy(mu)` = `mu`, i.e. pick what looks best
* `ucb(mu, var)` = `mu + beta * sqrt(var)` with beta = 2, i.e. pick what *could* be best

(`ucb` takes the **variance**, sigma². The repo's `EmbeddingMVEModel.get_means_and_vars` squares sigma before calling it.)

In [ ]:
from molpal.acquirer.metrics import greedy, ucb

names = ["A", "B", "C", "D"]
mu    = np.array([10.0, 9.5, 9.0, 6.0])
sigma = np.array([ 0.1, 0.2, 1.5, 0.3])

print("mol   mu   sigma  greedy   ucb")
for row in zip(names, mu, sigma, greedy(mu), ucb(mu, sigma**2)):
    print("  {}  {:5.1f}  {:4.1f}  {:6.1f}  {:5.1f}".format(*row))
print("greedy picks", names[np.argmax(greedy(mu))], "| ucb picks", names[np.argmax(ucb(mu, sigma**2))])

**Where does sigma come from?** One common trick is to train several models and measure
how much they **disagree**. Below, 5 linear models are each trained on a random resample
of the data. In the repo, `LTAllSurrogate` does the same with 5 neural nets, and
`SingleBackboneMVESurrogate` instead has a network head that predicts sigma directly
(the "MVE" head).

In [ ]:
class BootstrapEnsemble:
    def __init__(self, n_models=5, seed=0):
        self.n_models, self.rng = n_models, np.random.default_rng(seed)

    def fit(self, X, y):
        X1 = np.c_[X, np.ones(len(X))]
        self.ws = []
        for _ in range(self.n_models):
            pick = self.rng.integers(0, len(X), len(X))     # resample rows with replacement
            self.ws.append(np.linalg.lstsq(X1[pick], y[pick], rcond=None)[0])

    def predict(self, X):                                   # -> (mu, sigma)
        preds = np.stack([np.c_[X, np.ones(len(X))] @ w for w in self.ws])
        return preds.mean(axis=0), preds.std(axis=0)

ens = BootstrapEnsemble()
ens.fit(embeddings[train], -true_scores[train])
mu_t, sig_t = ens.predict(embeddings[test])
err = np.abs(mu_t - (-true_scores[test]))
print(f"mean |error| where sigma is in the lowest 20%: {err[sig_t < np.quantile(sig_t, .2)].mean():.2f}")
print(f"mean |error| where sigma is in the top 20%:    {err[sig_t > np.quantile(sig_t, .8)].mean():.2f}")

The error is usually larger where sigma is larger, so sigma carries real information. Now the question that matters: does **using** sigma (UCB) find more of the top molecules?

In [ ]:
def run_acq(acq, seed, n_rounds=5, init=100, batch=100):
    r = np.random.default_rng(seed)
    labeled = set(r.choice(N_POOL, init, replace=False).tolist())
    top = {int(s[4:]) for s in true_top_k}
    model, curve = BootstrapEnsemble(seed=seed), []
    for _ in range(n_rounds):
        idx = np.array(sorted(labeled))
        model.fit(embeddings[idx], -true_scores[idx])
        pool_idx = np.setdiff1d(np.arange(N_POOL), idx)
        mu, sigma = model.predict(embeddings[pool_idx])
        scores = ucb(mu, sigma**2) if acq == "ucb" else greedy(mu)
        labeled.update(pool_idx[np.argsort(scores)[::-1][:batch]].tolist())
        curve.append(len(labeled & top) / TOPK)
    return curve

for acq in ["greedy", "ucb"]:
    curves = np.array([run_acq(acq, s) for s in range(42, 47)])     # 5 seeds, like the repo
    m, s = curves.mean(0), curves.std(0)
    plt.errorbar(range(1, 6), m, yerr=s, marker="o", capsize=3, label=f"{acq}: {m[-1]:.0%} ± {s[-1]:.0%}")
plt.xlabel("round"); plt.ylabel("recall"); plt.legend(); plt.title("5 seeds each"); plt.show()

They come out about tied: the difference is within the seed-to-seed spread. That
matches the repo's real result. After 145 runs in `adaptive_acq/`, no β, schedule or
learned β controller beat plain greedy.

**Try it:** change `BootstrapEnsemble()` to `BootstrapEnsemble(n_models=1)` and look at
what sigma becomes. With one model there is nothing to disagree with, so sigma = 0 and
**UCB turns into greedy**. That was exactly one of the fusion bugs fixed on 2026-10-10:
`ucb` runs were identical to `greedy` runs.

## 7. The repo's real surrogates, and backbone fusion

A **backbone** is a big pretrained model that turns a molecule into an embedding:

* MoLFormer reads the SMILES text
* GROVER reads the molecular graph
* Uni-Mol reads the 3D shape

Each backbone runs **once** over the whole pool (`embed/`) and saves an `(N, D)` file.
During active learning only small nets are trained on top. That is why a campaign takes
minutes, not days.

The repo's main question is **one backbone or several fused together?** Method [3],
`EnsembleFusionSurrogate`, trains one small net per backbone and then combines them by
**rank**:

* rank each backbone's predictions (1 = best) and divide by n
* `mu` = −(mean rank), so molecules that all backbones like come first
* `sigma` ∝ the std of the ranks, i.e. how much the backbones **disagree**

Here is the real `combine()` on 5 molecules:

In [ ]:
from surrogates import SingleBackboneMVESurrogate, EnsembleFusionSurrogate

comps = np.array([      # each backbone's predicted (sign-flipped) score
    # grover molformer unimol
    [9.0,   8.8,   9.1],   # m0: everyone says great
    [5.0,   5.2,   4.9],   # m1: everyone says poor
    [9.5,   3.0,   4.0],   # m2: only grover loves it
    [7.0,   7.1,   6.9],   # m3: everyone says OK
    [2.0,   9.9,   2.5],   # m4: only molformer loves it
], dtype=np.float32)

fusion = EnsembleFusionSurrogate(dims={"grover": 1, "molformer": 1, "unimol": 1})
for beta in (0.2, 1.0):
    fusion._beta = beta
    mu, sigma = fusion.combine(comps)
    order_g = list(np.argsort(greedy(mu))[::-1])
    order_u = list(np.argsort(ucb(mu, sigma**2))[::-1])
    print(f"beta={beta}:  greedy order {['m%d' % i for i in order_g]}   ucb order {['m%d' % i for i in order_u]}")

At the default beta = 0.2 the disagreement bonus is too small to change the order. At
beta = 1.0, m2 and m4 (the "controversial" molecules) jump ahead of m3.

Now a small active-learning experiment with the real classes. We fake three backbones,
and each one is **blind to a different hidden feature**, just as a text model, a graph
model and a 3D model each miss different things. This cell takes about a minute on a CPU.

In [ ]:
import torch
torch.manual_seed(0)
r = np.random.default_rng(0)
N2, TOP2, INIT2, BATCH2, ROUNDS2, EPOCHS = 6_000, 60, 60, 60, 4, 30
hidden = r.normal(size=(N2, 6)).astype(np.float32)
y2 = (1.2*hidden[:, 0] - 0.8*hidden[:, 1] + 0.7*hidden[:, 2]*hidden[:, 3]
      + 0.5*np.sin(2*hidden[:, 4]) + 0.2*r.normal(size=N2)).astype(np.float32)   # higher = better
top2 = set(np.argsort(y2)[::-1][:TOP2].tolist())

def fake_backbone(dim, sees, seed):
    rr = np.random.default_rng(seed)
    return hidden[:, sees] @ rr.normal(size=(len(sees), dim)).astype(np.float32) \
           + 0.8 * rr.normal(size=(N2, dim)).astype(np.float32)

emb_dict = {"grover":    fake_backbone(32, [0, 1, 2, 3], 1),   # blind to feature 4
            "molformer": fake_backbone(24, [0, 1, 3, 4], 2),   # blind to feature 2
            "unimol":    fake_backbone(16, [0, 2, 3, 4], 3)}   # blind to feature 1
dims = {k: v.shape[1] for k, v in emb_dict.items()}
X_all = np.concatenate([emb_dict[k] for k in dims], axis=1)    # fusion takes them glued side by side
print({k: v.shape for k, v in emb_dict.items()}, "->", X_all.shape)

In [ ]:
def run_real(make_model, X, acq="greedy", seed=42):
    rr = np.random.default_rng(seed)
    labeled = set(rr.choice(N2, INIT2, replace=False).tolist())
    model = make_model()
    for _ in range(ROUNDS2):
        idx = np.array(sorted(labeled))
        model.fit(X[idx], y2[idx], epochs=EPOCHS)          # the repo's fit(X, y, epochs)
        pool_idx = np.setdiff1d(np.arange(N2), idx)
        mu, sigma = model.predict(X[pool_idx])              # the repo's predict(X) -> (mu, sigma)
        scores = ucb(mu, sigma**2) if acq == "ucb" else greedy(mu)
        labeled.update(pool_idx[np.argsort(scores)[::-1][:BATCH2]].tolist())
    return len(labeled & top2) / TOP2

import contextlib, io
results = {}
with contextlib.redirect_stdout(io.StringIO()):            # hide the per-backbone training log lines
    for k in dims:
        results[f"single: {k}"] = run_real(lambda k=k: SingleBackboneMVESurrogate(in_dim=dims[k]), emb_dict[k])
    results["fusion (greedy)"] = run_real(lambda: EnsembleFusionSurrogate(dims=dims, epochs=EPOCHS), X_all)
    results["fusion (ucb)"]    = run_real(lambda: EnsembleFusionSurrogate(dims=dims, epochs=EPOCHS), X_all, acq="ucb")

print(f"budget {INIT2 + ROUNDS2*BATCH2} of {N2:,} (random would get ~{(INIT2 + ROUNDS2*BATCH2)/N2:.0%})")
for name, rec in results.items():
    print(f"  {name:18s} recall {rec:.0%}")

Look at where fusion lands compared with the single backbones. Rank fusion gives every
backbone an **equal vote**, so the weak ones can drag the strong one down. That is why
the repo went on to **learn** the weights (`LearnedFusionSurrogate`, and `LTAllSurrogate`,
the current default).

On the real EnamineHTS data, frozen fusion reaches about 95% recall at the largest batch,
close to MPN and fine-tuned MoLFormer (about 95–98%), but 100–1000× cheaper. The biggest
improvements came from **better backbone embeddings** and **bug fixes**, not from cleverer
acquisition (`docs/adaptive_acq_results.md`, §1).

**Try it:** make one backbone nearly useless (`sees=[0]`) and rerun the two cells above.
How much does it hurt the fusion?

## 8. Results from real runs

Every `run_experiment.py` run writes a folder under `al-molecular/runs/`:

```
runs/<name>/
    iter_1/state.json        one round's summary (also used by --resume)
    iter_1/scores.pkl        every {smiles: score} docked so far
    history.json             all rounds  <- the learning curve
    all_explored_final.csv   every docked molecule, best first
```

Below we read the fixed fusion runs (`fusionfix_*`, 5 seeds each). Nothing is modified.

In [ ]:
RUNS = AL_REPO / "runs"
hist = json.loads((RUNS / "fusionfix_greedy_frac0.001_seed42" / "history.json").read_text())
for rec in hist:
    print(rec)

In [ ]:
from collections import defaultdict
curves = defaultdict(list)
for d in sorted(RUNS.glob("fusionfix_*_seed*")):
    f = d / "history.json"
    if not f.exists():
        continue
    h = json.loads(f.read_text())
    if len(h) < 5:                       # skip unfinished runs
        continue
    _, acq, frac, _ = d.name.split("_")
    curves[(frac, acq)].append([x["topk_recall"] for x in h])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, frac in zip(axes, ["frac0.001", "frac0.002", "frac0.004"]):
    for acq in ["greedy", "ucb"]:
        c = np.array(curves[(frac, acq)])
        ax.errorbar(range(1, 6), c.mean(0), yerr=c.std(0), marker="o", capsize=3,
                    label=f"{acq}: {c[:, -1].mean():.3f} ± {c[:, -1].std():.3f} (n={len(c)})")
    ax.set_title(f"batch = {float(frac[4:])*100:.1f}% of pool"); ax.set_xlabel("round"); ax.legend()
axes[0].set_ylabel("top-1000 recall")
plt.suptitle("EnamineHTS, fusion surrogate (fixed), real runs"); plt.tight_layout(); plt.show()

Two things to notice:

* **Batch size matters a lot.** A bigger batch means more molecules docked in total, so recall is higher.
* **Greedy vs UCB barely matters.** The curves sit on top of each other.

**Try it:** look at real molecules, the best ones found by one run.

In [ ]:
import pandas as pd
pd.read_csv(RUNS / "fusionfix_greedy_frac0.001_seed42" / "all_explored_final.csv").head(10)

## Where each piece lives in the real code

| What you did here | Real code |
|---|---|
| `oracle` dict | `load_oracle()`, `run_experiment.py:302` |
| `embeddings` | `load_embeddings()` → `molpal/featurizer.py` `EmbeddingFeaturizer` |
| `true_top_k`, `recall()` | `true_top_k_set()` :457, `MVEExplorer._recall()` |
| random init | `run_experiment.py:617` |
| sign flip | `self._sign = -1.0` :535, used at :647 |
| train | `self.model.train(...)` :649 |
| predict the pool | `_chunked_get_means_and_vars()` :657 (in 50k-row chunks, to save memory) |
| pick the batch | `np.argsort(scores)[::-1][:batch_size]` :664 |
| `greedy`, `ucb` | `molpal/acquirer/metrics.py` |
| surrogates | `surrogates.py`, chosen by `--surrogate` via `mve()` in `molpal/models/__init__.py:109` |

**Next:** open `run_experiment.py` at lines 638–683. It should now read like section 4 of
this notebook. After that, `06_al_molecular_tour/` covers the scale-up machinery
(memmaps, shard workers, resume).